# Auto-Fix Rules Summary Table

## Rule Application Order

**Yes, general fix rules are applied first, then column-specific rules.**

The fix process follows this order:
1. **Always trim whitespace first** (applied to all rule types)
2. Apply **general rules** for the rule type (enum, regex, length, phone, date, numeric)
3. Apply **column-specific rules** if applicable (these override or supplement general rules)
4. Validate the result against the pattern/rule

---

## Auto-Fix Rules Table

| Rule Type | General Fix Rules | Column Name | Column-Specific Rules | Example |
|-----------|-------------------|-------------|----------------------|---------|
| **ENUM** | 1. Trim whitespace 2. Exact match check 3. Case-insensitive match 4. Case normalization (upper/lower variants) | Entity Type Code | Float-to-integer normalization: "1.0" → "1", "2.0" → "2" (applied BEFORE general enum rules, then general rules applied) | "1.0" → "1", "ca" → "CA" |
| **ENUM** | Same as above | State Codes (Provider Business Mailing Address State Name, Provider Business Practice Location Address State Name, Provider License Number State Code_1, etc.) | None - uses general rules only | "ca" → "CA" |
| **ENUM** | Same as above | Primary Taxonomy Switch (Healthcare Provider Primary Taxonomy Switch_1, _2) | None - uses general rules only | "y" → "Y" |
| **REGEX** | 1. Trim whitespace 2. Normalize multiple spaces to single space 3. Pattern validation | NPI | Extract digits only. Remove all non-digit characters. Only fix if exactly 10 digits after cleaning. Cannot fix if not exactly 10 digits (no padding or truncation) | "123-456-7890" → "1234567890". "12345" → Cannot fix (too few digits). "1234567890123" → Cannot fix (too many digits) |
| **REGEX** | Same as above | Employer Identification Number (EIN) | Check for "<UNAVAIL>" first (case-insensitive) → normalize to "<UNAVAIL>". Extract digits, remove non-digits. Only fix if exactly 9 digits. Cannot fix if not exactly 9 digits (no truncation) | "12-3456789" → "123456789". "<unavail>" → "<UNAVAIL>". "12345" → Cannot fix (too few digits) |
| **REGEX** | Same as above | Postal Code (Provider Business Mailing Address Postal Code, Provider Business Practice Location Address Postal Code) | Remove all characters except digits and hyphens. Preserve hyphen for ZIP+4 format (12345-6789) | "ZIP: 12345" → "12345". "12345 6789" → "12345-6789" |
| **REGEX** | Same as above | Taxonomy Code (Healthcare Provider Taxonomy Code_1, _2) | Remove special characters (keep only letters and numbers). Convert to uppercase. Only fix if exactly 10 characters. Cannot fix if not exactly 10 characters (no truncation) | "12345-ABCDE" → "12345ABCDE". "12345ABCDEF" → Cannot fix (too long) |
| **REGEX** | Same as above | License Number (Provider License Number_1, _2) | Remove non-alphanumeric characters. Preserve case. Validate length is between 1-25 characters. Cannot fix if outside 1-25 character range (no truncation) | "LIC-12345" → "LIC12345". "LIC1234567890123456789012345" → Cannot fix (too long) |
| **REGEX** | Same as above | Name Fields (Provider First Name, Provider Last Name Legal Name) | None - uses general rules only | " John " → "John" |
| **REGEX** | Same as above | City Names (Provider Business Mailing Address City Name, Provider Business Practice Location Address City Name) | None - uses general rules only | " New York " → "New York" |
| **LENGTH** | 1. Trim whitespace 2. If length < min → Cannot fix 3. If length > max → Cannot fix (no truncation) | Provider Organization Name (Legal Business Name) | Min: 2, Max: 120. Uses general rules only | "A" → Cannot fix (too short). "A"*125 → Cannot fix (too long, no truncation) |
| **LENGTH** | Same as above | Address Fields (Provider First Line Business Mailing Address, Provider Second Line Business Mailing Address, Provider First Line Business Practice Location Address, Provider Second Line Business Practice Location Address) | Min: 1, Max: 80. Uses general rules only | "A"*85 → Cannot fix (too long, no truncation) |
| **LENGTH** | Same as above | Taxonomy Group (Healthcare Provider Taxonomy Group_1, _2) | Min: 1, Max: 50. Uses general rules only | "A"*55 → Cannot fix (too long, no truncation) |
| **LENGTH** | Same as above | Other Provider Identifier (Other Provider Identifier_1, _2) | Min: 1, Max: 50. Uses general rules only | Same as Taxonomy Group |
| **LENGTH** | Same as above | Other Provider Identifier Type Code (Other Provider Identifier Type Code_1, _2) | Min: 1, Max: 10. Uses general rules only | Same as above |
| **LENGTH** | Same as above | Other Provider Identifier Issuer (Other Provider Identifier Issuer_1, _2) | Min: 1, Max: 50. Uses general rules only | Same as above |
| **PHONE** | 1. Extract digits only (remove all non-digits) 2. If digit count within range → return 3. If > max digits → Cannot fix (no truncation) 4. If < min digits → Cannot fix (no padding) | Telephone Numbers (Provider Business Mailing Address Telephone Number, Provider Business Practice Location Address Telephone Number) | Min: 7 digits, Max: 20 digits. Uses general rules only | "(555) 123-4567" → "5551234567". "12345" → Cannot fix (too few digits). "555123456789012345678" → Cannot fix (too many digits, no truncation) |
| **PHONE** | Same as above | Fax Numbers (Provider Business Mailing Address Fax Number, Provider Business Practice Location Address Fax Number) | Min: 7 digits, Max: 20 digits. Uses general rules only | Same as telephone numbers |
| **DATE** | 1. Trim whitespace 2. Multi-format parsing: MM/DD/YYYY, YYYY-MM-DD, MM-DD-YYYY, YYYY/MM/DD, DD/MM/YYYY, MM.DD.YYYY, YYYY.MM.DD 3. Convert to MM/DD/YYYY 4. If not_future: true → reject future dates | Provider Enumeration Date | not_future: true. Uses general rules only | "2024-01-15" → "01/15/2024". "01/15/2025" → Cannot fix (future date) |
| **DATE** | Same as above | Last Update Date | not_future: true. Uses general rules only | Same as Provider Enumeration Date |
| **DATE** | Same as above | Certification Date | not_future: true. Uses general rules only | Same as Provider Enumeration Date |
| **NUMERIC** | 1. Remove thousand separators (commas) 2. Parse as float 3. Convert back to string | (Currently no numeric columns defined) | N/A | "1,234.56" → "1234.56" |
| **COMPLETENESS** | Row-by-row conditional imputation based on related fields | Entity Type Code | If Organization Name exists in the row → fill with "2" (Organization). If Individual Names (Last Name or First Name) exist in the row → fill with "1" (Individual). If neither exists → Cannot fix (requires manual review) | NULL (with Org Name) → "2". NULL (with Last Name) → "1". NULL (neither) → Cannot fix |

---

## Important Notes

1. **No Truncation**: Values that are too long cannot be fixed. Truncation is not performed as it would result in data loss.

2. **No Padding**: Values that are too short cannot be fixed. Padding with zeros or other characters is not performed as it would require domain knowledge to determine appropriate padding values.

3. **Completeness Fixes**: Only Entity Type Code completeness issues can be auto-fixed using row-by-row conditional logic based on related fields (Organization Name or Individual Names). All other completeness issues require manual review and domain expertise to determine appropriate values.

4. **Whitespace Handling**: All values are trimmed of leading/trailing whitespace before any other fixes are applied.

5. **Exact Match Required**: For numeric patterns (NPI, EIN), values must have exactly the required number of digits after cleaning. Values with too many or too few digits cannot be fixed.

6. **General Then Specific**: General rules are always applied first, then column-specific rules are checked and applied if applicable.

7. **Entity Type Code Completeness**: Missing Entity Type Code values are filled using row-by-row analysis. If a row has an Organization Name, the Entity Type Code is set to "2" (Organization). If a row has Individual Names (Last Name or First Name) but no Organization Name, the Entity Type Code is set to "1" (Individual). If a row has neither, the value cannot be automatically fixed.
